# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.crm_prd_info")

# Data Transformation

- TRIM the string
- prd_key UPPER + split
- Managed "null"
- Rename Columns

In [0]:
df.display()

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Upper + split prd_key

In [0]:
df = df.withColumn("prd_key", F.upper(col("prd_key")))

In [0]:
df = df.withColumn("category_id", col("prd_key").substr(1, 5))

In [0]:
df = df.withColumn("product_key", F.substring(col("prd_key"), 7, 100)).drop("prd_key")

## Nulls

"prd_cost" only 2 NULLs

- I decided to get the beginning of the "prd_nm" to group it and get the average cost of each LIKE product
ex:
"HL Road Frame - Black- 58" keep the first 12 caracters to have a better segmentation so it become "HL Road Fram" 

- Then round the column and changed the type to an INTEGER

In [0]:
df = df.withColumn(
    "prd_cost",
    F.when(
        col("prd_cost").isNull(),
        F.avg("prd_cost").over(Window.partitionBy(F.substring("prd_nm", 1, 12)))
    ).otherwise(col("prd_cost"))
)

In [0]:
df = df.withColumn("prd_cost", F.round(col("prd_cost")).cast("int"))

"prd_line" 
No line known so change it for "N/A" and see with business team to get the information

In [0]:
df = df.withColumn("prd_line", F.when(col("prd_line").isNull(), "N/A").otherwise(col("prd_line")))

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.crm_products")
)

# Check queries

In [0]:
%sql
SELECT 
  *
FROM data_lakehouse_project.silver.crm_products